<a href="https://colab.research.google.com/github/eren-darici/oil-blending-lp/blob/main/Project_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [522]:
!pip install pulp

import pulp as pl
import pandas as pd
import numpy as np

# Data

In [523]:
# Read data
# DATA_PATH = "data.xlsx"
DATA_PATH = "https://github.com/eren-darici/oil-blending-lp/blob/main/data.xlsx?raw=true"
master_data = pd.read_excel(DATA_PATH, sheet_name="master_data")
indexed_data = pd.read_excel(DATA_PATH, sheet_name="indexed_data")

price_data = pd.read_excel(DATA_PATH, sheet_name="price_data")
indexed_price_data = pd.read_excel(DATA_PATH, sheet_name="indexed_price_data")

In [524]:
indexed_data.set_index("Oil")["CCRi"]

,CCRi
Oil,
L,11.80
M,6.24
Gu,5.20
W.D,1.52
A.L,4.00
K,5.80
Q,4.04
B,6.33
Cond.,0.06


In [525]:
# Maps - dicts
li_map = (indexed_data.set_index("Oil")["Li"]/ 100).to_dict() # LPG yield (%)
nai_map = (indexed_data.set_index("Oil")["NAi"]/ 100).to_dict() # Naphtha yield (%)
ki_map = (indexed_data.set_index("Oil")["Ki"]/ 100).to_dict() # Kerosene yield (%)
gi_map = (indexed_data.set_index("Oil")["Gi"]/ 100).to_dict() # Gas oil yield (%)
fi_map = (indexed_data.set_index("Oil")["Fi"]/ 100).to_dict() # Fuel oil yield (%)

pi_map = indexed_data.set_index("Oil")["Pi"].to_dict() # Price $/BBL
cti_map = indexed_data.set_index("Oil")["CTi"].to_dict() #
cui_map = indexed_data.set_index("Oil")["CUi"].to_dict()

ccri_map = (indexed_data.set_index("Oil")["CCRi"] / 100).to_dict() # CCR wt %
mui_map = indexed_data.set_index("Oil")["mui"].to_dict() # Kinematic viscosity

ci_map = indexed_data.set_index("Oil")["Ci"].to_dict() # Conversion factor
gravityi_map = indexed_data.set_index("Oil")["gi"].to_dict() # Specific calculated gravity
si_map = (indexed_data.set_index("Oil")["Si"]).to_dict() # Salt content (ppm)
sui_map = (indexed_data.set_index("Oil")["SUi"]).to_dict() # Sulfur content (%)
ai_map = (indexed_data.set_index("Oil")["Ai"]).to_dict() # Acidity (mg KOH/gm)
wi_map = (indexed_data.set_index("Oil")["Wi"]).to_dict() # Wax content (%)

vi_map = indexed_data.set_index("Oil")["VI"].to_dict() # Viscosity Index

# SARA ratios - all in %
asi_map = (indexed_data.set_index("Oil")['SARA_ASi'] / 100).to_dict() # Asphaltenes
ri_map = (indexed_data.set_index("Oil")['SARA_Ri'] / 100).to_dict() # Resins
ari_map = (indexed_data.set_index("Oil")['SARA_ARi'] / 100).to_dict() # Aromatics
saturates_map = (indexed_data.set_index("Oil")['SARA_Pi'] / 100).to_dict() # Saturates

# Specifications of the Fuel Oil produced
sufi_map = (indexed_data.set_index("Oil")['SUFi']).to_dict() # Sulfur (%)
wfi_map = (indexed_data.set_index("Oil")['WFi']).to_dict() # Wax (%)
asfi_map = (indexed_data.set_index("Oil")['ASFi']).to_dict() # Asphaltanes (%)
vni_map = (indexed_data.set_index("Oil")['Vni']).to_dict() # Vanadium (ppm)
ni_map = (indexed_data.set_index("Oil")['Ni']).to_dict() # Nickel (ppm)

ti_map = indexed_data.set_index("Oil")["Ti"].to_dict() # Availability tons/d

fBBL_map = indexed_data.set_index("Oil")["F"].to_dict()

revenue_map = indexed_price_data.set_index("Yield")['Price'].to_dict()

comp_map = {
    "Li": li_map,
    "NAi": nai_map,
    "Ki": ki_map,
    "Gi": gi_map,
    "Fi": fi_map
}

# Decision Variables

In [526]:
# Create oils list to create decision variables
crude_oils = list(set(master_data['Oil']))

# Create the decision variables
var_oils = pl.LpVariable.dicts("oils", crude_oils, lowBound=0)
print(var_oils)

{'A.L': oils_A.L, 'L': oils_L, 'Gu': oils_Gu, 'K': oils_K, 'Q': oils_Q, 'Cond.': oils_Cond., 'W.D': oils_W.D, 'M': oils_M, 'B': oils_B, 'L.G': oils_L.G}


In [527]:
# LPG yields
# LPG yield from disttillation is x11, LPG yield from bed reformer x12, total x13 in the paper
# LPG yield decision variables
lpg_yield_from = ["distillation", "bed_reformer", "total"]
var_lpg_yields = pl.LpVariable.dicts("lpg", lpg_yield_from, lowBound=0)

In [528]:
# Product yields
# naphtha x14, kerosene x15, gas oil x16, fuel oil x17
products = ["naphtha", "kerosene", "gas oil", "fuel oil"]
var_product_yields = pl.LpVariable.dicts("product", products, lowBound=0)

In [529]:
# imported
# Naptha x18 , total naptha x19
# Reformer capacity x20, yield of reformate x21 (gasoline92)
# remaining available naptha after feeding the reformer x22
# gasoline 95 is x23
# gasoline80 is x24
otherx = ["imported_naptha", "total_naptha", "reformer_capacity", "gasoline92", "remaining_naptha", "gasoline95", "gasoline80"]
var_otherx = pl.LpVariable.dicts("otherx", otherx, lowBound=0)

# Objective Function

In [530]:
# Revenue function (revenue_i * li * xi)
revenue_components = ["Li", "NAi", "Ki", "Gi", "Fi"]

revenue = pl.lpSum(
    revenue_map[c] * globals()[f"{c.lower()}_map"][o] * var_oils[o]
    for c in revenue_components
    for o in crude_oils
) + 0.03 * revenue_map["Li"] * var_otherx["reformer_capacity"] + 0.97 * revenue_map["gasoline92"] * var_otherx["reformer_capacity"] + revenue_map["gasoline80"] * var_otherx["gasoline80"]


In [531]:
# Cost Function (conversion only applied for Pi (in bbls))
cost_components = ["Pi", "CTi", "CUi"]

cost = pl.lpSum(
    globals()[f"{c.lower()}_map"][o] * var_oils[o] *
    (fBBL_map[o] if c == "Pi" else 1)
    for c in cost_components
    for o in crude_oils
) + revenue_map["NAi"] * var_otherx["reformer_capacity"] + 1 *var_otherx["reformer_capacity"] + revenue_map["gasoline95"] * var_otherx["gasoline95"] + revenue_map["NAi"] * var_otherx["remaining_naptha"]

# Create the model

In [532]:
def create_model(model, test_case = 1, wax_conservative=0, gas92_demand=None, wax_fo=0):
  # Clear ALL existing constraints
  model.constraints.clear()

  # clear objective too
  model.objective = None

  # Objective function
  model += (revenue - cost), "objective"

  # Operating Capacity Constraint (17)
  model += pl.lpSum(ci_map[o] * var_oils[o] for o in crude_oils) <= 150000, "operatingCapacity"

  # Specific Gravity Constraint (18)
  model += pl.lpSum(ci_map[o] * gravityi_map[o] * var_oils[o] for o in crude_oils) == 0.867 * pl.lpSum(ci_map[o] * var_oils[o] for o in crude_oils) , "specificGravity"

  # Fuel Oil Production Constraint (19)
  model += pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils) >= 10600, "fuelOilProduction"

  # Salt Content Constraint (20)
  model += (pl.lpSum(si_map[o] * var_oils[o] for o in crude_oils) <= 30 * pl.lpSum(var_oils[o] for o in crude_oils)), "saltContent"

  # Sulfur Content Constraint (21)
  model += (pl.lpSum(sui_map[o] * var_oils[o] for o in crude_oils) <= 2.1 * pl.lpSum(var_oils[o] for o in crude_oils)), "sulfurContent"

  # Acidity Number Constraint (22)
  model += (pl.lpSum(ai_map[o] * var_oils[o] for o in crude_oils) <= 12.1 * pl.lpSum(var_oils[o] for o in crude_oils)), "acidityNumber"

  # Wax Content Constraint (23)
  if wax_conservative != 0:
    model += (pl.lpSum(wi_map[o] * var_oils[o] for o in crude_oils) <= (3.8 - wax_conservative) * pl.lpSum(var_oils[o] for o in crude_oils)), "waxNumber"

  else:
    model += (pl.lpSum(wi_map[o] * var_oils[o] for o in crude_oils) <= 3.8 * pl.lpSum(var_oils[o] for o in crude_oils)), "waxNumber"

  # CCR
  model += (pl.lpSum(ccri_map[o] * var_oils[o] for o in crude_oils) <= 6.1* pl.lpSum(var_oils[o] for o in crude_oils)), "ccri"

  # Viscosity Index Constraint (25)
  model += (pl.lpSum(vi_map[o] * var_oils[o] for o in crude_oils) <= 1.232 * pl.lpSum(var_oils[o] for o in crude_oils)), "viscosityIndex"

  # Compatability Constraint (26)
  model += (pl.lpSum(asi_map[o] * var_oils[o] for o in crude_oils) <= 0.35 * pl.lpSum(ri_map[o] * var_oils[o] for o in crude_oils)), "compatibility"

  # Collodial Instability Index Constraint (27) ??????
  model += pl.lpSum((saturates_map[o] + asi_map[o]) * var_oils[o] for o in crude_oils) <= 0.7 * pl.lpSum((ari_map[o] + ri_map[o]) * var_oils[o] for o in crude_oils), "CII"

  # Crude Oil Availability Constraint (28-37)
  for oil in crude_oils:
    model += var_oils[oil] <= ti_map[oil], f"{oil.replace(".", "")}_Availability"

  # Sulfur Content in Fuel Oil Constraint (38)
  model += pl.lpSum(sufi_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 3 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils), "sulfurContentFuelOil"

  # Wax Content in Fuel Oil Constraint (39)
  if wax_fo != 0:
    model += pl.lpSum(wfi_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= (7 - wax_fo) * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils), "waxContentFuelOil"
  else:
    model += pl.lpSum(wfi_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 7 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils), "waxContentFuelOil"

  # Asphaltane Content in Fuel Oil Constraint (40)
  model += pl.lpSum(asfi_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 5 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils), "asphaltaneContentFuelOil"

  # Vanadium Content in Fuel Oil Constraint (41)
  model += pl.lpSum(vni_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 70 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils), "vanadiumContentFuelOil"

  # Nickel Content in Fuel Oil Constraint (42)
  model += pl.lpSum(ni_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 40 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils), "nickelContentFuelOil"

  # LPG Yield from Distillation (43)
  model += var_lpg_yields["distillation"] == pl.lpSum(li_map[o] * var_oils[o] for o in crude_oils), "lpgFromDistillation"

  # LPG Yield from Reformer (44)
  model += var_lpg_yields["bed_reformer"] - 0.03 * var_otherx["reformer_capacity"] == 0, "lpgFromReformer"

  # Total LPG Yield (45)
  model += var_lpg_yields["total"] - var_lpg_yields["distillation"] - var_lpg_yields["bed_reformer"] == 0, "totalLpg"

  # Naphtha Yield from Distillation (46)
  model += var_product_yields["naphtha"] == pl.lpSum(nai_map[o] * var_oils[o] for o in crude_oils), "naphthaFromDistillation"

  # Kerosene Yield from Distillation (47)
  model += var_product_yields["kerosene"] == pl.lpSum(ki_map[o] * var_oils[o] for o in crude_oils), "keroseneFromDistillation"

  # Gas Oil Yield from Distillation (48)
  model += var_product_yields["gas oil"] == pl.lpSum(gi_map[o] * var_oils[o] for o in crude_oils), "gasOilFromDistillation"

  # Fuel Oil Yield from Distillation (49)
  model += var_product_yields["fuel oil"] == pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils), "fuelOilFromDistillation"

  # Total Naphtha Available (50)
  model += var_otherx["total_naptha"] == var_product_yields['naphtha'] + var_otherx["imported_naptha"], "totalNaphtha"

  # Imported Naphtha Constraint (51)
  model += var_otherx["imported_naptha"] <= 5000, "importedNaphtha"

  # Naphtha Distribution Constraint (52)
  model += var_otherx["total_naptha"]  == var_otherx["reformer_capacity"] + var_otherx["remaining_naptha"], "naphthaDistribution"

  # Fixed Bed Reformer Capacity Constraint (53)
  model += var_otherx["reformer_capacity"] <= 5000, "fixedBedReformer"

  # Reformate Yield from Reformer (54)
  model += 1.031 * var_otherx["gasoline92"] - var_otherx["reformer_capacity"] == 0, "reformateFromReformer"

  # Gasoline 80 Blending (55)
  model += (var_otherx["gasoline95"] * 0.4 == 0.6 * var_otherx["remaining_naptha"]), "gasoline80Blending"

  # Gasoline 80 Yield (56)
  model += (var_otherx["gasoline80"] == var_otherx['remaining_naptha'] + var_otherx['gasoline95']), "gasoline80Yield"


  # Gasoline 80-92 Demands (57-58)
  model += var_otherx["gasoline80"] >= 4000, "gasoline80Demand"

  if gas92_demand is not None:
    model += var_otherx["gasoline92"] >= gas92_demand, "gasoline92Demand"
  else:
    model += var_otherx["gasoline92"] >= 4500, "gasoline92Demand"

  # Imported Gasoline 95 Constraint (59)
  model += var_otherx["gasoline95"] <= 10000, "importedGasoline95"

  if test_case in (2, 3, 4):
    # Extra demands case
    # LPG demand (60)
    model += var_lpg_yields["total"] >= 400, "lpgDemand"

    # Kerosene demand (61)
    model += var_product_yields["kerosene"] >= 1500, "keroseneDemand"

    # Gas oil demand (62)
    model += var_product_yields["gas oil"] >= 3500, "gasOilDemand"


  if test_case in (3, 4):
    # Case 2 +  AL availability (63)
    model += var_oils["A.L"] <= 10000, "ALAvailability"

  if test_case == 4:
    # Case 3 + M and Gu demand
    # M demand (64)
    model += var_oils["M"] >= 400, "MDemand"

    # Gu demand (65)
    model += var_oils["Gu"] >= 2000, "GuDemand"

  return model


# Utilities for Printing Purposes

In [533]:
model_map = {
    # Crude oils (X1–X10)
    "X1 (L)": var_oils["L"],
    "X2 (M)": var_oils["M"],
    "X3 (Gu)": var_oils["Gu"],
    "X4 (W.D)": var_oils["W.D"],
    "X5 (A.L)": var_oils["A.L"],
    "X6 (K)": var_oils["K"],
    "X7 (Q)": var_oils["Q"],
    "X8 (B)": var_oils["B"],
    "X9 (Cond.)": var_oils["Cond."],
    "X10 (L.G)": var_oils["L.G"],

    # LPG yields (X11–X13)
    "X11 (LPG distillation)": var_lpg_yields["distillation"],
    "X12 (LPG reformer)": var_lpg_yields["bed_reformer"],
    "X13 (Total LPG)": var_lpg_yields["total"],

    "X14 (Naphtha yield)": var_product_yields["naphtha"],
    "X15 (Kerosene yield)": var_product_yields["kerosene"],
    "X16 (Gas oil yield)": var_product_yields["gas oil"],
    "X17 (Fuel oil yield)": var_product_yields["fuel oil"],

    # Other variables (X18–X23)
    "X18 (Imported naphtha)": var_otherx["imported_naptha"],
    "X19 (Total naphtha)": var_otherx["total_naptha"],

    "X20 (Reformer capacity)": var_otherx["reformer_capacity"],
    "X21 (Gasoline 92)": var_otherx["gasoline92"],
    "X22 (Remaining naphtha)": var_otherx["remaining_naptha"],
    "X23 (Gasoline 95)": var_otherx["gasoline95"],
    "X24 (Gasoline 80)": var_otherx["gasoline80"],
}

# Solve and Print

In [534]:
print(pl.listSolvers(onlyAvailable=True))
solver = pl.getSolver("PULP_CBC_CMD")

['PULP_CBC_CMD', 'HiGHS']


In [535]:
!rm -rf models/
!mkdir models/

In [536]:
results = {}
for tc in range(1, 5):
  case_results = {}
  # Create the model
  model = pl.LpProblem("OilBlending", pl.LpMaximize)

  # Add constraints
  model = create_model(model, tc)

  model.solve(solver)

  case_results["status"] = pl.LpStatus[model.status]
  case_results["objective"] = pl.value(model.objective)
  for k, v in model_map.items():
    k = k.split(" ")[0]
    case_results[k] = v.value()
  results[tc] = case_results


In [537]:
results = {}
gas92_demands = [3500, 4000, 5000, 5500]
for tc in range(1, 5):
  for g92 in gas92_demands:
    case_results = {}
    # Create the model
    model = pl.LpProblem("OilBlending", pl.LpMaximize)

    # Add constraints
    model = create_model(model, tc, gas92_demand=g92)

    model.solve(solver)


    case_results["status"] = pl.LpStatus[model.status]
    case_results["objective"] = pl.value(model.objective)
    for k, v in model_map.items():
      k = k.split(" ")[0]
      case_results[k] = v.value()
    results[f"tc{tc}_g92-{g92}"] = case_results


In [538]:
results = {}
wax_cons = [-.5, -.3, -.1,.5, .3, .1]
for tc in range(1, 5):
  for wc in wax_cons:
    case_results = {}
    # Create the model
    model = pl.LpProblem("OilBlending", pl.LpMaximize)

    # Add constraints
    model = create_model(model, tc, wax_conservative=wc)


    model.solve(solver)
    case_results["status"] = pl.LpStatus[model.status]
    case_results["objective"] = pl.value(model.objective)
    for k, v in model_map.items():
      k = k.split(" ")[0]
      case_results[k] = v.value()
    results[f"tc{tc}_wc_{wc}"] = case_results


In [539]:
results = {}
wax_f = [-5, -3, -2, 5, 3, 2]
for tc in range(1, 5):
  for wf in wax_f:
    case_results = {}
    # Create the model
    model = pl.LpProblem("OilBlending", pl.LpMaximize)

    # Add constraints
    model = create_model(model, tc, wax_fo=wf)
    maple_text = convert_model(model)
    Path(f"models/waxInFuelOil_model_tc{tc}_wf{wf}.mpl").write_text(maple_text, encoding="utf-8")
    model.solve()
    case_results["status"] = pl.LpStatus[model.status]
    case_results["objective"] = pl.value(model.objective)
    for k, v in model_map.items():
      k = k.split(" ")[0]
      case_results[k] = v.value()
    results[f"tc{tc}_wf-{wf}"] = case_results


In [541]:
results

{'tc1_wf--5': {'status': 'Optimal',
  'objective': 372848.80221240455,
  'X1': 2628.4128,
  'X2': 0.0,
  'X3': 0.0,
  'X4': 0.0,
  'X5': 13300.0,
  'X6': 0.0,
  'X7': 0.0,
  'X8': 1036.794,
  'X9': 1106.1273,
  'X10': 2600.0,
  'X11': 251.43062,
  'X12': 150.0,
  'X13': 401.43062,
  'X14': 2822.0217,
  'X15': 1954.8944,
  'X16': 3687.8122,
  'X17': 11851.819,
  'X18': 3777.9783,
  'X19': 6600.0,
  'X20': 5000.0,
  'X21': 4849.6605,
  'X22': 1600.0,
  'X23': 2400.0,
  'X24': 4000.0},
 'tc1_wf--3': {'status': 'Optimal',
  'objective': 372848.80221240455,
  'X1': 2628.4128,
  'X2': 0.0,
  'X3': 0.0,
  'X4': 0.0,
  'X5': 13300.0,
  'X6': 0.0,
  'X7': 0.0,
  'X8': 1036.794,
  'X9': 1106.1273,
  'X10': 2600.0,
  'X11': 251.43062,
  'X12': 150.0,
  'X13': 401.43062,
  'X14': 2822.0217,
  'X15': 1954.8944,
  'X16': 3687.8122,
  'X17': 11851.819,
  'X18': 3777.9783,
  'X19': 6600.0,
  'X20': 5000.0,
  'X21': 4849.6605,
  'X22': 1600.0,
  'X23': 2400.0,
  'X24': 4000.0},
 'tc1_wf--2': {'status':

In [542]:
def results_dict_to_df(results):
    df = pd.DataFrame(results)  # columns = test cases
    df.columns = [f"Test Case {c}" for c in df.columns]

    return df